# AdaptiveMath-AI — Sequential and Deep Model Comparators

**Notebook 04 of 06**

Compact mechanism-oriented neural comparators are evaluated under a common tensor, optimizer and split contract. They are proxies and are not presented as canonical reproductions of the original architectures.

## 1. Reproducible tensors, sequence histories, and evaluation contract

Create identical current-question, prior-sequence, and tabular inputs for all neural families. Map train entities to vocabularies with index 0 reserved for unseen entities, build right-aligned histories of length 30, standardize a pre-declared set of safe numeric features from train only, and configure MPS when available.

Leakage control: Every sequence contains interactions strictly before the target; current correctness and current answer choice never enter inputs. Entity vocabularies and standardization parameters are train-only.

In [1]:
from pathlib import Path
import copy, json, math, os, random, time, textwrap, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display
import psutil
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,f1_score,precision_score,recall_score,roc_auc_score,average_precision_score,matthews_corrcoef,cohen_kappa_score,log_loss,brier_score_loss,confusion_matrix)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Subset

mpl.rcParams.update({"font.family":"Times New Roman","font.size":18,"axes.titlesize":20,"axes.labelsize":19,"xtick.labelsize":18,"ytick.labelsize":18,"legend.fontsize":18,"figure.titlesize":20,"axes.spines.top":False,"axes.spines.right":False,"savefig.dpi":350,"figure.dpi":350})
PALETTE={"blue":"#2F6B8F","orange":"#C77C2B","green":"#4F7F3A","purple":"#6D5A8D","red":"#B45A55","gray":"#6E7378","light_gray":"#E7EAED","teal":"#4A8C8A","gold":"#C9A227","black":"#222222","white":"#FFFFFF"}
SEED=20260713
def seed_all(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_all(SEED)
cwd=Path.cwd().resolve(); ROOT=next((p for p in [cwd,*cwd.parents] if (p/"notebooks").is_dir() and (p/"README.md").exists()),None)
if ROOT is None: raise FileNotFoundError("Run from the repository root or the notebooks directory.")
MODEL_DIR=ROOT/"models/sota_model_artifacts"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
device=torch.device("mps" if torch.backends.mps.is_available() else "cpu"); torch.set_num_threads(max(1,min(8,os.cpu_count() or 1)))
data=pd.read_parquet(ROOT/"dataset/processed/modeling_dataset.parquet").reset_index(drop=True)
assert {"missing_confidence","history_eligible","Confidence","answer_value","correct_answer"}.isdisjoint(data.columns)
train_mask=data.split_id.eq("train"); train_rows=data[train_mask]
q_values=np.sort(train_rows.question_id.unique()); s_values=np.sort(train_rows.primary_subject_id.unique()); u_values=np.sort(train_rows.user_id.unique())
q_map={int(v):i+1 for i,v in enumerate(q_values)}; s_map={int(v):i+1 for i,v in enumerate(s_values)}; u_map={int(v):i+1 for i,v in enumerate(u_values)}
q_idx=data.question_id.map(q_map).fillna(0).astype("int32").to_numpy(); s_idx=data.primary_subject_id.map(s_map).fillna(0).astype("int16").to_numpy(); u_idx=data.user_id.map(u_map).fillna(0).astype("int32").to_numpy()
# Mechanism-oriented numeric features fixed before validation.
deep_features=[c for c in ["prior_interaction_count","prior_cumulative_accuracy","rolling_accuracy_5","rolling_accuracy_20","ewm_accuracy","prior_subject_accuracy","question_difficulty","question_popularity_past","rasch_ability","rasch_question_difficulty","elo_probability","bkt_mastery","subject_mastery_probability","days_since_previous_interaction","days_since_previous_subject_interaction","recent_learning_trend","recent_error_streak","recent_correct_streak","historical_group_performance","historical_quiz_performance","historical_scheme_performance","student_question_similarity","collaborative_state_norm","subject_depth"] if c in data]
train_numeric=data.loc[train_mask,deep_features].replace([np.inf,-np.inf],np.nan); med=train_numeric.median().fillna(0); mean=train_numeric.fillna(med).mean(); std=train_numeric.fillna(med).std().replace(0,1)
X=((data[deep_features].replace([np.inf,-np.inf],np.nan).fillna(med)-mean)/std).clip(-8,8).astype("float32").to_numpy()
y=data.is_correct.astype("float32").to_numpy()
# Strictly prior histories.
L=30; n=len(data); hist_q=np.zeros((n,L),dtype="int32"); hist_s=np.zeros((n,L),dtype="int16"); hist_r=np.zeros((n,L),dtype="int8"); hist_gap=np.zeros((n,L),dtype="float32"); hist_mask=np.zeros((n,L),dtype="bool")
gaps=data.days_since_previous_interaction.fillna(0).clip(0,365).to_numpy("float32")
for _,idxs in data.groupby("user_id",sort=False).groups.items():
 eligible_prior=[]
 for i in np.asarray(list(idxs),dtype=int):
  prior=np.asarray(eligible_prior[-L:],dtype=int); length=len(prior)
  if length:
   hist_q[i,-length:]=q_idx[prior]; hist_s[i,-length:]=s_idx[prior]; hist_r[i,-length:]=y[prior].astype("int8"); hist_gap[i,-length:]=np.log1p(gaps[prior]); hist_mask[i,-length:]=True
  if data.at[i,"split_id"]!="external_unseen_question": eligible_prior.append(i)
assert np.all(hist_mask.sum(1)==data.prior_interaction_count.clip(upper=L).to_numpy())
tensors=TensorDataset(torch.from_numpy(X.copy()),torch.from_numpy(hist_q.copy()),torch.from_numpy(hist_s.astype("int64",copy=True)),torch.from_numpy(hist_r.astype("int64",copy=True)),torch.from_numpy(hist_gap.copy()),torch.from_numpy(hist_mask.copy()),torch.from_numpy(q_idx.astype("int64",copy=True)),torch.from_numpy(s_idx.astype("int64",copy=True)),torch.from_numpy(u_idx.astype("int64",copy=True)),torch.from_numpy(y.copy()))
indices={s:np.flatnonzero(data.split_id.eq(s).to_numpy()) for s in ["train","validation","temporal_test","external_unseen_student","external_unseen_question"]}
hashes=pd.util.hash_pandas_object(data.loc[indices["train"],"interaction_id"].astype(str)+"deep_dev",index=False).to_numpy("uint64")
dev_indices=indices["train"][np.argsort(hashes)[:100_000]]
val_hash=pd.util.hash_pandas_object(data.loc[indices["validation"],"interaction_id"].astype(str)+"early_stop",index=False).to_numpy("uint64"); early_val_indices=indices["validation"][np.argsort(val_hash)[:20_000]]
print(f"Device: {device}; tensor rows={n:,}; sequence length={L}; numeric features={len(deep_features)}")
print(f"Development train={len(dev_indices):,}; full train={len(indices['train']):,}; early-stop validation={len(early_val_indices):,}")
display(pd.DataFrame({"split":indices.keys(),"rows":[len(v) for v in indices.values()]}))

def save_and_show_figure(fig,filename): path=ROOT/"figures"/filename; fig.savefig(path,dpi=350,bbox_inches="tight",facecolor="white"); plt.show(); print(f"Saved figure: {path}"); return path
def annotate_bars(ax,fmt="{:.3f}"):
 for p in ax.patches: ax.text(p.get_x()+p.get_width(),p.get_y()+p.get_height()/2," "+fmt.format(p.get_width()),va="center",fontsize=18,clip_on=False)
def wrap_labels(labels,width=28): return ["\n".join(textwrap.wrap(str(x),width)) for x in labels]
def plot_horizontal_metric_ranking(frame,label_col,metric_col,color=PALETTE["purple"]):
 z=frame.sort_values(metric_col); fig,ax=plt.subplots(figsize=(11,max(5,.5*len(z)))); ax.barh(wrap_labels(z[label_col]),z[metric_col],color=color); annotate_bars(ax); return fig,ax
def place_legend_below(ax,ncol=3): ax.legend(loc="upper center",bbox_to_anchor=(.5,-.16),ncol=ncol,frameon=False,fontsize=18)
def safe_p(p): return np.clip(np.asarray(p,float),1e-6,1-1e-6)
def cal_error(y0,p,bins=15):
 p=safe_p(p); ids=np.minimum((p*bins).astype(int),bins-1); ece=0.; mce=0.
 for b in range(bins):
  m=ids==b
  if m.any(): gap=abs(y0[m].mean()-p[m].mean()); ece+=m.mean()*gap; mce=max(mce,gap)
 return ece,mce
def metric_row(name,y0,p,split):
 p=safe_p(p); z=(p>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y0,z,labels=[0,1]).ravel(); ece,mce=cal_error(y0,p)
 try: auc=roc_auc_score(y0,p); pr=average_precision_score(y0,p)
 except Exception: auc=pr=np.nan
 try:
  xx=logit(p).reshape(-1,1); recal=LogisticRegression(C=1e6,solver="lbfgs",max_iter=300).fit(xx,y0); slope=float(recal.coef_[0,0]); intercept=float(recal.intercept_[0])
 except Exception: slope=intercept=np.nan
 return {"Model":name,"Split":split,"N":len(y0),"Accuracy":accuracy_score(y0,z),"Balanced_Accuracy":balanced_accuracy_score(y0,z),"Macro_F1":f1_score(y0,z,average="macro",zero_division=0),"Weighted_F1":f1_score(y0,z,average="weighted",zero_division=0),"Precision":precision_score(y0,z,zero_division=0),"Recall":recall_score(y0,z,zero_division=0),"Specificity":tn/max(tn+fp,1),"ROC_AUC":auc,"PR_AUC":pr,"MCC":matthews_corrcoef(y0,z),"Kappa":cohen_kappa_score(y0,z),"Log_Loss":log_loss(y0,p,labels=[0,1]),"Brier_Score":brier_score_loss(y0,p),"ECE":ece,"MCE":mce,"FPR":fp/max(fp+tn,1),"FNR":fn/max(fn+tp,1),"Calibration_Slope":slope,"Calibration_Intercept":intercept,"Confusion_Matrix":f"[[{tn},{fp}],[{fn},{tp}]]"}

Device: mps; tensor rows=449,557; sequence length=30; numeric features=24
Development train=100,000; full train=281,459; early-stop validation=20,000


,split,rows
0,train,281459
1,validation,60750
2,temporal_test,63209
3,external_unseen_student,22687
4,external_unseen_question,21452


## 2. Compact neural architecture library

Test distinct neural mechanisms without implying exact reproduction of canonical literature architectures. Implement compact GRU, static key–value attention, current-question attention, time-aware and subject-aware attention, Transformer, graph-smoothed subject, deep-tabular, and collaborative proxies. The registry names state the implemented mechanism; `Canonical_Reimplementation=False` is an explicit evidence boundary.

Leakage control: Every model receives the same prior-only history tensors and train-fitted vocabularies.

In [2]:
NQ=len(q_map)+1; NS=len(s_map)+1; NU=len(u_map)+1; NF=len(deep_features); D=24
class DKT(nn.Module):
 def __init__(self):
  super().__init__(); self.inter=nn.Embedding(2*NQ,D,padding_idx=0); self.q=nn.Embedding(NQ,D,padding_idx=0); self.gru=nn.GRU(D,32,batch_first=True); self.out=nn.Sequential(nn.Linear(32+D,32),nn.ReLU(),nn.Dropout(.1),nn.Linear(32,1))
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  token=hq+hr*NQ; _,h=self.gru(self.inter(token)); return self.out(torch.cat([h[-1],self.q(q)],1)).squeeze(1)
class DKVMN(nn.Module):
 def __init__(self):
  super().__init__(); self.key=nn.Embedding(NQ,D,padding_idx=0); self.value=nn.Embedding(2*NQ,D,padding_idx=0); self.out=nn.Sequential(nn.Linear(2*D,32),nn.ReLU(),nn.Linear(32,1))
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  query=self.key(q); keys=self.key(hq); values=self.value(hq+hr*NQ); score=(keys*query[:,None,:]).sum(-1)/math.sqrt(D); score=score.masked_fill(~hm,-1e4); att=torch.softmax(score,1); memory=(att[:,:,None]*values).sum(1); return self.out(torch.cat([query,memory],1)).squeeze(1)
class SAKT(nn.Module):
 def __init__(self,time_aware=False,subject_aware=False):
  super().__init__(); self.time_aware=time_aware; self.subject_aware=subject_aware; self.inter=nn.Embedding(2*NQ,D,padding_idx=0); self.q=nn.Embedding(NQ,D,padding_idx=0); self.s=nn.Embedding(NS,D,padding_idx=0); self.pos=nn.Embedding(L,D); self.att=nn.MultiheadAttention(D,4,batch_first=True,dropout=.1); self.norm=nn.LayerNorm(D); self.out=nn.Sequential(nn.Linear(D,24),nn.ReLU(),nn.Linear(24,1)); self.last_attention=None
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  pos=torch.arange(L,device=hq.device)[None,:]; kv=self.inter(hq+hr*NQ)+self.pos(pos)
  if self.time_aware: kv=kv+torch.tanh(hg[:,:,None])*0.1
  if self.subject_aware: kv=kv+self.s(hs)
  query=(self.q(q)+(self.s(s) if self.subject_aware else 0))[:,None,:]; empty=~hm.any(1); pad=(~hm).clone(); pad[:,-1]=False
  z,w=self.att(query,kv,kv,key_padding_mask=pad,need_weights=True); z=torch.where(empty[:,None,None],torch.zeros_like(z),z); self.last_attention=w.detach(); return self.out(self.norm(z[:,0,:]+query[:,0,:])).squeeze(1)
class TransformerKT(nn.Module):
 def __init__(self):
  super().__init__(); self.inter=nn.Embedding(2*NQ,D,padding_idx=0); self.q=nn.Embedding(NQ,D,padding_idx=0); self.pos=nn.Embedding(L,D); layer=nn.TransformerEncoderLayer(D,4,48,batch_first=True,dropout=.1); self.enc=nn.TransformerEncoder(layer,2,enable_nested_tensor=False); self.out=nn.Sequential(nn.Linear(2*D,32),nn.ReLU(),nn.Linear(32,1))
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  pos=torch.arange(L,device=hq.device)[None,:]; pad=(~hm).clone(); pad[:,-1]=False; z=torch.nan_to_num(self.enc(self.inter(hq+hr*NQ)+self.pos(pos),src_key_padding_mask=pad)); pooled=(z*hm[:,:,None]).sum(1)/hm.sum(1,keepdim=True).clamp_min(1); return self.out(torch.cat([pooled,self.q(q)],1)).squeeze(1)
# Fixed normalized subject graph for GKT.
graph=pd.read_parquet(ROOT/"dataset/processed/question_subject_graph.parquet"); A=np.eye(NS,dtype="float32")
for row in graph.itertuples():
 a=s_map.get(int(row.source_subject_id),0); b=s_map.get(int(row.target_subject_id),0)
 if a and b: A[a,b]=A[b,a]=1
A=A/np.maximum(A.sum(1,keepdims=True),1)
class GKT(nn.Module):
 def __init__(self):
  super().__init__(); self.s=nn.Embedding(NS,D,padding_idx=0); self.register_buffer("adj",torch.from_numpy(A)); self.out=nn.Sequential(nn.Linear(2*D+NF,48),nn.ReLU(),nn.Dropout(.1),nn.Linear(48,1))
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  graph_emb=self.adj@self.s.weight; current=graph_emb[s]; hist=(graph_emb[hs]*hm[:,:,None]).sum(1)/hm.sum(1,keepdim=True).clamp_min(1); return self.out(torch.cat([current,hist,x],1)).squeeze(1)
class ResidualMLP(nn.Module):
 def __init__(self):
  super().__init__(); self.inp=nn.Linear(NF,64); self.b1=nn.Sequential(nn.LayerNorm(64),nn.ReLU(),nn.Dropout(.1),nn.Linear(64,64)); self.b2=nn.Sequential(nn.LayerNorm(64),nn.ReLU(),nn.Linear(64,64)); self.out=nn.Linear(64,1)
 def forward(self,x,*args):
  z=torch.relu(self.inp(x)); z=z+self.b1(z); z=z+self.b2(z); return self.out(z).squeeze(1)
class CompactTabNet(nn.Module):
 def __init__(self):
  super().__init__(); self.mask=nn.Sequential(nn.Linear(NF,NF),nn.Sigmoid()); self.trunk=nn.Sequential(nn.Linear(NF,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU()); self.out=nn.Linear(32,1)
 def forward(self,x,*args): return self.out(self.trunk(x*self.mask(x))).squeeze(1)
class FTTransformer(nn.Module):
 def __init__(self):
  super().__init__(); self.weight=nn.Parameter(torch.randn(NF,16)*.05); self.bias=nn.Parameter(torch.zeros(NF,16)); self.cls=nn.Parameter(torch.zeros(1,1,16)); layer=nn.TransformerEncoderLayer(16,4,32,batch_first=True,dropout=.1); self.enc=nn.TransformerEncoder(layer,1,enable_nested_tensor=False); self.out=nn.Linear(16,1)
 def forward(self,x,*args):
  tok=x[:,:,None]*self.weight[None,:,:]+self.bias[None,:,:]; cls=self.cls.expand(len(x),-1,-1); z=self.enc(torch.cat([cls,tok],1)); return self.out(z[:,0]).squeeze(1)
class TabTransformer(nn.Module):
 def __init__(self,saint=False):
  super().__init__(); self.saint=saint; self.q=nn.Embedding(NQ,16,padding_idx=0); self.s=nn.Embedding(NS,16,padding_idx=0); self.u=nn.Embedding(NU,16,padding_idx=0); self.num=nn.Linear(NF,16); layer=nn.TransformerEncoderLayer(16,4,32,batch_first=True,dropout=.15 if saint else .1); self.enc=nn.TransformerEncoder(layer,2 if saint else 1,enable_nested_tensor=False); self.out=nn.Sequential(nn.Linear(16,16),nn.ReLU(),nn.Linear(16,1))
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  tokens=torch.stack([self.q(q),self.s(s),self.u(u),self.num(x)],1); z=self.enc(tokens); return self.out(z.mean(1)).squeeze(1)
class NeuralMF(nn.Module):
 def __init__(self,mlp=False):
  super().__init__(); self.ue=nn.Embedding(NU,D,padding_idx=0); self.qe=nn.Embedding(NQ,D,padding_idx=0); self.mlp=mlp; self.out=nn.Sequential(nn.Linear(2*D,32),nn.ReLU(),nn.Linear(32,1)) if mlp else None
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  ue,qe=self.ue(u),self.qe(q); return self.out(torch.cat([ue,qe],1)).squeeze(1) if self.mlp else (ue*qe).sum(1)/math.sqrt(D)
class DeepFM(nn.Module):
 def __init__(self):
  super().__init__(); self.ue=nn.Embedding(NU,16,padding_idx=0); self.qe=nn.Embedding(NQ,16,padding_idx=0); self.se=nn.Embedding(NS,16,padding_idx=0); self.deep=nn.Sequential(nn.Linear(NF+48,64),nn.ReLU(),nn.Dropout(.1),nn.Linear(64,1)); self.bias=nn.Parameter(torch.zeros(1))
 def forward(self,x,hq,hs,hr,hg,hm,q,s,u):
  fields=torch.stack([self.ue(u),self.qe(q),self.se(s)],1); fm=.5*((fields.sum(1)**2-fields.pow(2).sum(1)).sum(1)); deep=self.deep(torch.cat([x,fields.flatten(1)],1)).squeeze(1); return deep+fm/math.sqrt(16)+self.bias
registry={
 "DKT-inspired GRU proxy":("knowledge_tracing",DKT),"Static key-value attention proxy":("knowledge_tracing",DKVMN),"SAKT-inspired attention proxy":("knowledge_tracing",lambda:SAKT()),"Time-aware SAKT proxy":("knowledge_tracing",lambda:SAKT(time_aware=True)),"Subject-time SAKT proxy":("knowledge_tracing",lambda:SAKT(time_aware=True,subject_aware=True)),"Compact Transformer KT proxy":("knowledge_tracing",TransformerKT),"Subject-graph embedding proxy":("knowledge_tracing",GKT),
 "Gated MLP proxy":("deep_tabular",CompactTabNet),"Compact numeric-token Transformer proxy":("deep_tabular",FTTransformer),"Compact entity-token Transformer proxy":("deep_tabular",TabTransformer),"Deeper entity-token Transformer proxy":("deep_tabular",lambda:TabTransformer(saint=True)),"Residual MLP":("deep_tabular",ResidualMLP),
 "Neural MF proxy":("collaborative",NeuralMF),"DeepFM-style proxy":("collaborative",DeepFM),"Student-question embedding MLP":("collaborative",lambda:NeuralMF(mlp=True))}
display(pd.DataFrame([{"Model":k,"Family":v[0],"Canonical_Reimplementation":False,"Permitted_Claim":"compact mechanism-oriented comparator"} for k,v in registry.items()]))

,Model,Family,Canonical_Reimplementation,Permitted_Claim
0,DKT-inspired GRU proxy,knowledge_tracing,False,compact mechanism-oriented comparator
1,Static key-value attention proxy,knowledge_tracing,False,compact mechanism-oriented comparator
2,SAKT-inspired attention proxy,knowledge_tracing,False,compact mechanism-oriented comparator
3,Time-aware SAKT proxy,knowledge_tracing,False,compact mechanism-oriented comparator
4,Subject-time SAKT proxy,knowledge_tracing,False,compact mechanism-oriented comparator
5,Compact Transformer KT proxy,knowledge_tracing,False,compact mechanism-oriented comparator
6,Subject-graph embedding proxy,knowledge_tracing,False,compact mechanism-oriented comparator
7,Gated MLP proxy,deep_tabular,False,compact mechanism-oriented comparator
8,Compact numeric-token Transformer proxy,deep_tabular,False,compact mechanism-oriented comparator
9,Compact entity-token Transformer proxy,deep_tabular,False,compact mechanism-oriented comparator


## 3. Common training loop and development screening

Compare architectures under the same optimizer, deterministic subset, early stopping, clipping, and sequence truncation. Train up to four epochs with AdamW, gradient-norm clipping, and patience-two validation stopping; record loss, time, parameters, and validation probabilities. This is a bounded mechanism screen, not a definitive SOTA benchmark.

Leakage control: Early stopping observes the designated validation subset only. Test and external loaders are not called until family winners are selected.

In [3]:
BATCH=512
def loader(idxs,shuffle=False,batch=BATCH): return DataLoader(Subset(tensors,list(map(int,idxs))),batch_size=batch,shuffle=shuffle,num_workers=0)
def move(batch): return [t.to(device) for t in batch]
def evaluate_loss(model,idxs):
 model.eval(); total=0.; count=0; loss_fn=nn.BCEWithLogitsLoss(reduction="sum")
 with torch.no_grad():
  for batch in loader(idxs,False):
   *features,target=move(batch); logits=model(*features); total+=float(loss_fn(logits,target).cpu()); count+=len(target)
 return total/max(count,1)
def train_model(model,train_idxs,seed=SEED,max_epochs=4):
 seed_all(seed); model=model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); loss_fn=nn.BCEWithLogitsLoss(); best=None; best_loss=np.inf; patience=2; waited=0; history=[]; start=time.perf_counter()
 for epoch in range(max_epochs):
  model.train(); running=0.; count=0
  for batch in loader(train_idxs,True):
   *features,target=move(batch); opt.zero_grad(set_to_none=True); logits=model(*features); loss=loss_fn(logits,target); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); running+=float(loss.detach().cpu())*len(target); count+=len(target)
  vloss=evaluate_loss(model,early_val_indices); history.append({"epoch":epoch+1,"train_loss":running/max(count,1),"validation_loss":vloss})
  if vloss<best_loss-1e-4: best_loss=vloss; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; waited=0
  else:
   waited+=1
   if waited>=patience: break
 model.load_state_dict(best); model=model.to(device); return model,pd.DataFrame(history),time.perf_counter()-start
def predict_model(model,idxs,capture_attention=False):
 model.eval(); probs=[]; attention=[]; start=time.perf_counter()
 with torch.no_grad():
  for batch in loader(idxs,False):
   *features,target=move(batch); logits=model(*features); probs.append(torch.sigmoid(logits).detach().cpu().numpy())
   if capture_attention and hasattr(model,"last_attention") and model.last_attention is not None: attention.append(model.last_attention.cpu().numpy())
 return np.concatenate(probs),time.perf_counter()-start,attention
screen=[]; models={}; histories={}; efficiency=[]; validation_predictions={}
for name,(family,ctor) in registry.items():
 print(f"Training {name} on development cohort...")
 model,history,seconds=train_model(ctor(),dev_indices); pred,infer,_=predict_model(model,indices["validation"]); auc=roc_auc_score(y[indices["validation"]],pred); ll=log_loss(y[indices["validation"]],safe_p(pred)); params=sum(p.numel() for p in model.parameters() if p.requires_grad)
 path=MODEL_DIR/(name.lower().replace(" ","_").replace("+","plus").replace("/","_")+"_development.pt"); torch.save({"state_dict":{k:v.cpu() for k,v in model.state_dict().items()},"model":name,"family":family,"sequence_length":L,"features":deep_features,"training_rows":len(dev_indices),"seed":SEED},path)
 screen.append({"Model":name,"Family":family,"Validation_ROC_AUC":auc,"Validation_Log_Loss":ll,"Training_Rows":len(dev_indices),"Epochs":len(history),"Parameters":params,"Artifact":str(path.relative_to(ROOT))})
 efficiency.append({"Model":name,"Family":family,"Training_Time_Seconds":seconds,"Inference_Time_Seconds":infer,"Evaluated_Interactions":len(indices["validation"]),"Latency_Milliseconds_Per_Interaction":1000*infer/len(indices["validation"]),"Throughput_Interactions_Per_Second":len(indices["validation"])/max(infer,1e-9),"Model_Size_Bytes":path.stat().st_size,"Parameter_Count":params,"Training_Rows":len(dev_indices),"Device":str(device)})
 models[name]=model; histories[name]=history; validation_predictions[name]=pred
 print(f"{name}: validation ROC-AUC={auc:.4f}, log loss={ll:.4f}, epochs={len(history)}, params={params:,}")
 if device.type=="mps": torch.mps.empty_cache()
screen=pd.DataFrame(screen); display(screen.sort_values(["Family","Validation_ROC_AUC"],ascending=[True,False])); print("Development screening completed without accessing test or external labels.")

Training DKT-inspired GRU proxy on development cohort...


DKT-inspired GRU proxy: validation ROC-AUC=0.6537, log loss=0.6337, epochs=4, params=1,705,617
Training Static key-value attention proxy on development cohort...


Static key-value attention proxy: validation ROC-AUC=0.6204, log loss=0.6427, epochs=4, params=1,699,793
Training SAKT-inspired attention proxy on development cohort...


SAKT-inspired attention proxy: validation ROC-AUC=0.6188, log loss=0.6406, epochs=4, params=1,708,897
Training Time-aware SAKT proxy on development cohort...


Time-aware SAKT proxy: validation ROC-AUC=0.6186, log loss=0.6407, epochs=4, params=1,708,897
Training Subject-time SAKT proxy on development cohort...


...

Neural MF proxy: validation ROC-AUC=0.4986, log loss=0.7981, epochs=4, params=688,704
Training DeepFM-style proxy on development cohort...


DeepFM-style proxy: validation ROC-AUC=0.7072, log loss=0.6808, epochs=4, params=468,482
Training Student-question embedding MLP on development cohort...


Student-question embedding MLP: validation ROC-AUC=0.6117, log loss=0.6409, epochs=4, params=690,305


,Model,Family,Validation_ROC_AUC,Validation_Log_Loss,Training_Rows,Epochs,Parameters,Artifact
13,DeepFM-style proxy,collaborative,0.707198,0.680804,100000,4,468482,models/sota_model_artifacts/deepfm-style_proxy...
14,Student-question embedding MLP,collaborative,0.611665,0.640877,100000,4,690305,models/sota_model_artifacts/student-question_e...
12,Neural MF proxy,collaborative,0.498644,0.798071,100000,4,688704,models/sota_model_artifacts/neural_mf_proxy_de...
7,Gated MLP proxy,deep_tabular,0.768891,0.547441,100000,4,4313,models/sota_model_artifacts/gated_mlp_proxy_de...
11,Residual MLP,deep_tabular,0.768155,0.547716,100000,4,10241,models/sota_model_artifacts/residual_mlp_devel...
8,Compact numeric-token Transformer proxy,deep_tabular,0.767074,0.549338,100000,4,3025,models/sota_model_artifacts/compact_numeric-to...
10,Deeper entity-token Transformer proxy,deep_tabular,0.763262,0.554014,100000,4,468881,models/sota_model_artifacts/deeper_entity-toke...
9,Compact entity-token Transformer proxy,deep_tabular,0.762174,0.554895,100000,4,466657,models/sota_model_artifacts/compact_entity-tok...
6,Subject-graph embedding proxy,knowledge_tracing,0.768209,0.549038,100000,4,10465,models/sota_model_artifacts/subject-graph_embe...
0,DKT-inspired GRU proxy,knowledge_tracing,0.653676,0.633718,100000,4,1705617,models/sota_model_artifacts/dkt-inspired_gru_p...


Development screening completed without accessing test or external labels.


## 4. Validation-selected family winners and full feasible retraining

Give each major neural family a strong final representative while respecting compute limits. Select the validation-best knowledge-tracing, deep-tabular, and collaborative architectures, retrain those three on all 281k training interactions with the same early-stopping protocol, and replace their development predictions.

Leakage control: Selection is validation-only. Temporal test and external labels remain unopened until all three winner artifacts are frozen.

In [4]:
winners=screen.sort_values(["Family","Validation_ROC_AUC"],ascending=[True,False]).groupby("Family").head(1)
for row in winners.itertuples():
 name=row.Model; family=row.Family; print(f"Full feasible retraining: {name} ({family})")
 ctor=registry[name][1]; model,history,seconds=train_model(ctor(),indices["train"]); pred,infer,_=predict_model(model,indices["validation"]); validation_predictions[name]=pred; models[name]=model; histories[name]=history
 path=MODEL_DIR/(name.lower().replace(" ","_").replace("+","plus").replace("/","_")+"_full_train.pt"); params=sum(p.numel() for p in model.parameters() if p.requires_grad); torch.save({"state_dict":{k:v.cpu() for k,v in model.state_dict().items()},"model":name,"family":family,"sequence_length":L,"features":deep_features,"training_rows":len(indices["train"]),"seed":SEED},path)
 screen.loc[screen.Model==name,["Validation_ROC_AUC","Validation_Log_Loss","Training_Rows","Epochs","Parameters","Artifact"]]=[roc_auc_score(y[indices["validation"]],pred),log_loss(y[indices["validation"]],safe_p(pred)),len(indices["train"]),len(history),params,str(path.relative_to(ROOT))]
 efficiency=[r for r in efficiency if r["Model"]!=name]; efficiency.append({"Model":name,"Family":family,"Training_Time_Seconds":seconds,"Inference_Time_Seconds":infer,"Evaluated_Interactions":len(indices["validation"]),"Latency_Milliseconds_Per_Interaction":1000*infer/len(indices["validation"]),"Throughput_Interactions_Per_Second":len(indices["validation"])/max(infer,1e-9),"Model_Size_Bytes":path.stat().st_size,"Parameter_Count":params,"Training_Rows":len(indices["train"]),"Device":str(device)})
 print(f"Saved full model: {path} ({path.stat().st_size/2**20:.2f} MiB); validation ROC-AUC={roc_auc_score(y[indices['validation']],pred):.4f}")
 if device.type=="mps": torch.mps.empty_cache()
hyper=[]
for row in screen.itertuples(): hyper.append({"Model":row.Model,"Family":row.Family,"Sequence_Length":L,"Batch_Size":BATCH,"Max_Epochs":4,"Early_Stopping_Patience":2,"Optimizer":"AdamW","Learning_Rate":1e-3,"Gradient_Clip_Norm":1.0,"Mixed_Precision":False,"Training_Rows":int(row.Training_Rows),"Validation_Selected_Family_Winner":bool(row.Model in set(winners.Model)),"Parameters":int(row.Parameters),"Seed":SEED,"Evidence_Boundary":"compact proxy; not canonical reproduction"})
hyper=pd.DataFrame(hyper); hyper.to_csv(ROOT/"artifacts/sota_hyperparameter_summary.csv",index=False)
display(screen.sort_values("Validation_ROC_AUC",ascending=False)); display(hyper[hyper.Validation_Selected_Family_Winner])
print(f"Saved table: {ROOT/'artifacts/sota_hyperparameter_summary.csv'}")

Full feasible retraining: DeepFM-style proxy (collaborative)


Saved full model: <repository_root>/models/sota_model_artifacts/deepfm-style_proxy_full_train.pt (1.79 MiB); validation ROC-AUC=0.7349
Full feasible retraining: Gated MLP proxy (deep_tabular)


Saved full model: <repository_root>/models/sota_model_artifacts/gated_mlp_proxy_full_train.pt (0.02 MiB); validation ROC-AUC=0.7705
Full feasible retraining: Subject-graph embedding proxy (knowledge_tracing)


Saved full model: <repository_root>/models/sota_model_artifacts/subject-graph_embedding_proxy_full_train.pt (0.36 MiB); validation ROC-AUC=0.7693


,Model,Family,Validation_ROC_AUC,Validation_Log_Loss,Training_Rows,Epochs,Parameters,Artifact
7,Gated MLP proxy,deep_tabular,0.770470,0.546396,281459,4,4313,models/sota_model_artifacts/gated_mlp_proxy_fu...
6,Subject-graph embedding proxy,knowledge_tracing,0.769334,0.547411,281459,4,10465,models/sota_model_artifacts/subject-graph_embe...
11,Residual MLP,deep_tabular,0.768155,0.547716,100000,4,10241,models/sota_model_artifacts/residual_mlp_devel...
8,Compact numeric-token Transformer proxy,deep_tabular,0.767074,0.549338,100000,4,3025,models/sota_model_artifacts/compact_numeric-to...
10,Deeper entity-token Transformer proxy,deep_tabular,0.763262,0.554014,100000,4,468881,models/sota_model_artifacts/deeper_entity-toke...
9,Compact entity-token Transformer proxy,deep_tabular,0.762174,0.554895,100000,4,466657,models/sota_model_artifacts/compact_entity-tok...
13,DeepFM-style proxy,collaborative,0.734933,0.607668,281459,4,468482,models/sota_model_artifacts/deepfm-style_proxy...
0,DKT-inspired GRU proxy,knowledge_tracing,0.653676,0.633718,100000,4,1705617,models/sota_model_artifacts/dkt-inspired_gru_p...
5,Compact Transformer KT proxy,knowledge_tracing,0.642342,0.632964,100000,4,1710257,models/sota_model_artifacts/compact_transforme...
1,Static key-value attention proxy,knowledge_tracing,0.620439,0.642677,100000,4,1699793,models/sota_model_artifacts/static_key-value_a...


,Model,Family,Sequence_Length,Batch_Size,Max_Epochs,Early_Stopping_Patience,Optimizer,Learning_Rate,Gradient_Clip_Norm,Mixed_Precision,Training_Rows,Validation_Selected_Family_Winner,Parameters,Seed,Evidence_Boundary
6,Subject-graph embedding proxy,knowledge_tracing,30,512,4,2,AdamW,0.001,1.0,False,281459,True,10465,20260713,compact proxy; not canonical reproduction
7,Gated MLP proxy,deep_tabular,30,512,4,2,AdamW,0.001,1.0,False,281459,True,4313,20260713,compact proxy; not canonical reproduction
13,DeepFM-style proxy,collaborative,30,512,4,2,AdamW,0.001,1.0,False,281459,True,468482,20260713,compact proxy; not canonical reproduction


Saved table: <repository_root>/artifacts/sota_hyperparameter_summary.csv


## 5. Frozen temporal, cold-start, short-history, and attention evaluation

Compare neural families under future interactions and entity cold start, and expose how SAKT distributes attention over prior lags. Generate probabilities for validation, temporal test, unseen students, and unseen questions; apply the complete metric contract; reuse external unseen-student predictions for exact 5/10/20/50-history stress targets.

Leakage control: No result in this cell changes an architecture, threshold, feature, or winner.

In [5]:
predictions={"validation":validation_predictions}; timing={}
for split in ["temporal_test","external_unseen_student","external_unseen_question"]:
 predictions[split]={}
 for name,model in models.items():
  p,seconds,_=predict_model(model,indices[split]); predictions[split][name]=p; timing[(split,name)]=seconds
# Attention-by-lag diagnostic for the compact SAKT-inspired proxy.
attention_model="SAKT-inspired attention proxy"
_,_,att_batches=predict_model(models[attention_model],indices["validation"][:min(10_000,len(indices["validation"]))],capture_attention=True)
if att_batches:
 att=np.concatenate(att_batches,axis=0)
 if att.ndim==3: att=att[:,0,:]
 mean_att=np.nanmean(att,axis=0); attention_summary=pd.DataFrame({"lag":np.arange(L,0,-1),"mean_attention_weight":mean_att})
else: attention_summary=pd.DataFrame({"lag":np.arange(L,0,-1),"mean_attention_weight":np.nan})
attention_summary.to_csv(ROOT/"artifacts/sota_attention_diagnostics.csv",index=False)
metric_tables={}; prediction_tables={}; suffix={"validation":"validation","temporal_test":"test","external_unseen_student":"external_unseen_student","external_unseen_question":"external_unseen_question"}
for split,models_pred in predictions.items():
 rows=[]; frames=[]; base=data.loc[indices[split],["interaction_id","answer_id","user_id","question_id","is_correct","split_id","prior_interaction_count"]].reset_index(drop=True)
 for name,pred in models_pred.items(): rows.append(metric_row(name,y[indices[split]].astype(int),pred,split)); f=base.copy(); f["Model"]=name; f["Predicted_Probability"]=safe_p(pred); frames.append(f)
 mt=pd.DataFrame(rows).sort_values(["ROC_AUC","Log_Loss"],ascending=[False,True]); pt=pd.concat(frames,ignore_index=True); metric_tables[split]=mt; prediction_tables[split]=pt
 mp=ROOT/"artifacts"/f"sota_{suffix[split]}_metrics.csv"; pp=ROOT/"artifacts"/f"sota_predictions_{suffix[split]}.parquet"; mt.to_csv(mp,index=False); pt.to_parquet(pp,index=False,compression="zstd"); print(f"Saved table: {mp}"); print(f"Saved predictions: {pp} ({pp.stat().st_size/2**20:.1f} MiB)"); display(mt[["Model","N","ROC_AUC","Log_Loss","Brier_Score","ECE"]].head(8))
stress=pd.read_parquet(ROOT/"dataset/processed/stress_short_history.parquet"); ext_base=data.loc[indices["external_unseen_student"],["answer_id"]].reset_index(drop=True); stress_rows=[]
for cap,g in stress.groupby("history_cap"):
 for name,pred in predictions["external_unseen_student"].items():
  lookup=dict(zip(ext_base.answer_id,safe_p(pred))); pp=np.array([lookup[a] for a in g.answer_id]); stress_rows.append({**metric_row(name,g.is_correct.to_numpy(),pp,"stress_short_history"),"History_Cap":int(cap)})
short=pd.DataFrame(stress_rows); short.to_csv(ROOT/"artifacts/sota_short_history_metrics.csv",index=False)
efficiency=pd.DataFrame(efficiency); efficiency.to_csv(ROOT/"artifacts/sota_training_efficiency.csv",index=False)
print(f"Saved table: {ROOT/'artifacts/sota_short_history_metrics.csv'}"); print(f"Saved table: {ROOT/'artifacts/sota_training_efficiency.csv'}"); print(f"Saved attention diagnostics: {ROOT/'artifacts/sota_attention_diagnostics.csv'}")
display(short.sort_values(["History_Cap","ROC_AUC"],ascending=[True,False]).groupby("History_Cap").head(4)[["History_Cap","Model","N","ROC_AUC","Log_Loss"]]); display(efficiency.sort_values("Training_Time_Seconds")); display(attention_summary.head())

Saved table: <repository_root>/artifacts/sota_validation_metrics.csv
Saved predictions: <repository_root>/artifacts/sota_predictions_validation.parquet (10.0 MiB)


,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE
7,Gated MLP proxy,60750,0.770470,0.546396,0.184417,0.015009
6,Subject-graph embedding proxy,60750,0.769334,0.547411,0.184804,0.013413
11,Residual MLP,60750,0.768155,0.547716,0.185058,0.006666
8,Compact numeric-token Transformer proxy,60750,0.767074,0.549338,0.185605,0.013544
10,Deeper entity-token Transformer proxy,60750,0.763262,0.554014,0.187296,0.020191
9,Compact entity-token Transformer proxy,60750,0.762174,0.554895,0.187716,0.023037
13,DeepFM-style proxy,60750,0.734933,0.607668,0.204083,0.076095
0,DKT-inspired GRU proxy,60750,0.653676,0.633718,0.220151,0.052135


Saved table: <repository_root>/artifacts/sota_test_metrics.csv
Saved predictions: <repository_root>/artifacts/sota_predictions_test.parquet (10.4 MiB)


,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE
7,Gated MLP proxy,63209,0.770289,0.555659,0.188080,0.019087
6,Subject-graph embedding proxy,63209,0.769740,0.556257,0.188363,0.021257
8,Compact numeric-token Transformer proxy,63209,0.767917,0.557661,0.189088,0.022067
11,Residual MLP,63209,0.767624,0.556950,0.188828,0.012047
10,Deeper entity-token Transformer proxy,63209,0.763876,0.562910,0.190982,0.026238
9,Compact entity-token Transformer proxy,63209,0.762690,0.564343,0.191624,0.031839
13,DeepFM-style proxy,63209,0.735322,0.617853,0.207782,0.078280
0,DKT-inspired GRU proxy,63209,0.638902,0.655068,0.228928,0.060138


Saved table: <repository_root>/artifacts/sota_external_unseen_student_metrics.csv
Saved predictions: <repository_root>/artifacts/sota_predictions_external_unseen_student.parquet (3.5 MiB)


...

,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE
6,Subject-graph embedding proxy,21452,0.737406,0.578538,0.197691,0.042994
7,Gated MLP proxy,21452,0.737358,0.579096,0.197776,0.041978
11,Residual MLP,21452,0.734938,0.576267,0.196837,0.020716
8,Compact numeric-token Transformer proxy,21452,0.732998,0.579869,0.198201,0.034410
13,DeepFM-style proxy,21452,0.726888,0.611472,0.206729,0.083131
9,Compact entity-token Transformer proxy,21452,0.726243,0.598420,0.203646,0.058979
10,Deeper entity-token Transformer proxy,21452,0.725910,0.588083,0.201004,0.042914
0,DKT-inspired GRU proxy,21452,0.668688,0.622539,0.215771,0.036776


Saved table: <repository_root>/artifacts/sota_short_history_metrics.csv
Saved table: <repository_root>/artifacts/sota_training_efficiency.csv
Saved attention diagnostics: <repository_root>/artifacts/sota_attention_diagnostics.csv


,History_Cap,Model,N,ROC_AUC,Log_Loss
10,5,Deeper entity-token Transformer proxy,271,0.767507,0.518795
8,5,Compact numeric-token Transformer proxy,271,0.759295,0.526096
9,5,Compact entity-token Transformer proxy,271,0.754265,0.539389
7,5,Gated MLP proxy,271,0.751846,0.529924
22,10,Gated MLP proxy,271,0.772441,0.530055
26,10,Residual MLP,271,0.771824,0.534330
21,10,Subject-graph embedding proxy,271,0.767070,0.535940
23,10,Compact numeric-token Transformer proxy,271,0.761822,0.536099
36,20,Subject-graph embedding proxy,271,0.786433,0.520057
41,20,Residual MLP,271,0.774831,0.528047


,Model,Family,Training_Time_Seconds,Inference_Time_Seconds,Evaluated_Interactions,Latency_Milliseconds_Per_Interaction,Throughput_Interactions_Per_Second,Model_Size_Bytes,Parameter_Count,Training_Rows,Device
11,Student-question embedding MLP,collaborative,16.412867,1.586944,60750,0.026123,38281.132004,2765497,690305,100000,mps
10,Neural MF proxy,collaborative,16.584339,1.752133,60750,0.028842,34672.026300,2757829,688704,100000,mps
9,Residual MLP,deep_tabular,19.791246,1.800698,60750,0.029641,33736.921470,46641,10241,100000,mps
1,Static key-value attention proxy,knowledge_tracing,20.912777,1.741529,60750,0.028667,34883.138363,6803537,1699793,100000,mps
6,Compact numeric-token Transformer proxy,deep_tabular,22.106391,2.047916,60750,0.033711,29664.306259,20068,3025,100000,mps
7,Compact entity-token Transformer proxy,deep_tabular,22.185527,2.196556,60750,0.036157,27656.936226,1875757,466657,100000,mps
3,Time-aware SAKT proxy,knowledge_tracing,22.277696,2.086374,60750,0.034344,29117.500678,6841857,1708897,100000,mps
2,SAKT-inspired attention proxy,knowledge_tracing,23.158448,1.862114,60750,0.030652,32624.208051,6842401,1708897,100000,mps
4,Subject-time SAKT proxy,knowledge_tracing,26.099594,1.966894,60750,0.032377,30886.262229,6841897,1708897,100000,mps
8,Deeper entity-token Transformer proxy,deep_tabular,27.163946,2.062096,60750,0.033944,29460.315520,1888614,468881,100000,mps


,lag,mean_attention_weight
0,30,0.035439
1,29,0.031680
2,28,0.030453
3,27,0.032623
4,26,0.033906


## 6. Seed stability

The validation-best neural proxy is refitted with three random seeds on the same training partition to quantify initialization sensitivity.

In [6]:
best_name=screen.sort_values("Validation_ROC_AUC",ascending=False).iloc[0].Model
seed_rows=[{"Model":best_name,"Seed":SEED,"Training_Rows":len(indices["train"]),"Validation_ROC_AUC":roc_auc_score(y[indices["validation"]],validation_predictions[best_name])}]
for seed in [SEED+1,SEED+2]:
 extra_model,extra_hist,extra_seconds=train_model(registry[best_name][1](),indices["train"],seed=seed); extra_pred,_,_=predict_model(extra_model,indices["validation"])
 seed_rows.append({"Model":best_name,"Seed":seed,"Training_Rows":len(indices["train"]),"Validation_ROC_AUC":roc_auc_score(y[indices["validation"]],extra_pred)})
seed_stability=pd.DataFrame(seed_rows); assert seed_stability.Training_Rows.nunique()==1; seed_stability.to_csv(ROOT/"artifacts/sota_seed_stability.csv",index=False); display(seed_stability)
print("All registered neural methods are compact mechanism-oriented proxies rather than canonical reproductions.")

,Model,Seed,Training_Rows,Validation_ROC_AUC
0,Gated MLP proxy,20260713,281459,0.770470
1,Gated MLP proxy,20260714,281459,0.768214
2,Gated MLP proxy,20260715,281459,0.769901


,artifact,exists
0,artifacts/sota_validation_metrics.csv,True
1,artifacts/sota_test_metrics.csv,True
2,artifacts/sota_external_unseen_student_metrics.csv,True
3,artifacts/sota_external_unseen_question_metrics.csv,True
4,artifacts/sota_short_history_metrics.csv,True
5,artifacts/sota_predictions_validation.parquet,True
6,artifacts/sota_predictions_test.parquet,True
7,artifacts/sota_predictions_external_unseen_studen...,True
8,artifacts/sota_predictions_external_unseen_questi...,True
9,artifacts/sota_training_efficiency.csv,True


Evidence boundary: all registered neural methods are compact mechanism-oriented proxies; no result is a claim against canonical paper implementations.
